In [ ]:
import pandas as pd
df = pd.read_csv("/content/blind_all_keywords_final.csv")
df

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 코랩 konlpy 실행
!curl -s https://raw.githubusercontent.com/teddylee777/machine-learning/master/99-Misc/01-Colab/mecab-colab.sh | bash

# Mecab 설치 후 Google Drive에 복사
!cp -r /usr/local/lib/mecab /content/drive/MyDrive/mecab
!cp -r /usr/local/etc/mecabrc /content/drive/MyDrive/mecab

In [ ]:
! pip install kiwipiepy
import pandas as pd
import re
from konlpy.tag import Okt
from kiwipiepy import Kiwi

# 통합본문 만들기

In [ ]:
import pandas as pd

# 1. 데이터 불러오기 (이미 불러온 상태라면 생략 가능)
# df = pd.read_csv('레몬테라스_층간소음_2023부터.xlsx - Sheet1.csv')

# 2. 식별 정보 제거 (익명성 보장)
if '작성자' in df.columns:
    df = df.drop(columns=['작성자'])

# 3. 결측치(NaN)를 빈 문자열로 변환
df['제목'] = df['제목'].fillna('')
df['본문'] = df['본문'].fillna('')
df['댓글'] = df['댓글'].fillna('')

# 4. '제목', '본문', '댓글'을 합쳐서 '통합본문' 컬럼 생성
# 각 텍스트가 자연스럽게 이어지도록 사이에 공백(' ')을 추가합니다.
df['통합본문'] = df['제목'] + ' ' + df['본문'] + ' ' + df['댓글']


In [ ]:
df

# 중복글 확인 및 제거 -> 확인해보니 중복 없네



In [ ]:
# 중복 검사 기준 열
dup_subset = ['제목', '날짜', '본문']

# 중복된 모든 행(원본 포함)을 찾아서 필터링
# keep=False 옵션을 주면 중복된 쌍(세트) 전체를 모두 띄워줍니다.
duplicated_rows = df[df.duplicated(subset=dup_subset, keep=False)]

# 눈으로 확인하기 쉽게 제목과 날짜순으로 정렬
duplicated_rows = duplicated_rows.sort_values(by=['제목', '날짜'])

# 중복 데이터가 총 몇 개인지 확인
print(f"중복된 전체 행 개수: {len(duplicated_rows)}개")

# 중복된 데이터 직접 출력해서 보기 (전체 열 확인)
print(duplicated_rows[dup_subset])

In [ ]:
# 중복을 검사할 기준 열 지정
dup_subset = ['제목', '날짜', '본문']

# 지정한 열들을 기준으로 중복 데이터 제거 (첫 번째 행만 유지)
df = df.drop_duplicates(subset=dup_subset, keep='first')

# 중복 제거 후 빈 인덱스를 깔끔하게 재정렬
df = df.reset_index(drop=True)

# 결과 확인
df

# 동의어 유의어 처리

In [ ]:
import pandas as pd
import re


# 2. 추가 노이즈 단어 리스트 (알바, 견적, 정보요청 등)
EXTRA_NOISE_WORDS = ['알바', '견적', '정보좀주세요', '공유부탁', '쪽지주세요', '비댓', '비댓글']
noise_pattern = '|'.join(EXTRA_NOISE_WORDS)

# [실행] 노이즈 행 제거
before_len = len(df)
df = df[~df['통합본문'].str.contains(noise_pattern, na=False, case=False)]
print(f"✅ 노이즈 행 제거 완료: {before_len}개 -> {len(df)}개")

# 3. 동의어/유의어 사전 정의
synonym_dict = {

  # 소음 관련
    '층소': '층간소음', '층소충': '층간소음', '발망치': '층간소음', '쌩발망치소리': '층간소음',
    '쿵쾅': '층간소음', '쿵쾅쿵쾅': '층간소음', '쿵쿵': '층간소음', '쿵쾅거림': '층간소음',
    '우당탕': '층간소음', '우당탕탕': '층간소음', '우다다': '층간소음', '우다다다': '층간소음',

    # 이웃 지칭
    '위층': '윗집', '윗층': '윗집', '아래층': '아랫집', '밑에집': '아랫집',

    # 아이/연령 관련
    '애': '아이', '애기': '아이', '아기': '아이', '아가': '아이', '애들': '아이들', '꼬맹이': '아이들',
    '첫째': '아이', '둘째': '아이', '아들': '아이', '딸': '아이', '아들램': '아이',
    '초딩': '초등학생', '초등': '초등학생', '초1': '초등학생', '초2': '초등학생', '초3': '초등학생',
    '초4': '초등학생', '초5': '초등학생', '초6': '초등학생', '저학년': '초등학생', '고학년': '초등학생',
    '중딩': '중학생', '중1': '중학생', '중2': '중학생', '중3': '중학생',
    '고딩': '고등학생', '고1': '고등학생', '고3': '고등학생', '수험생': '고등학생',
    '노인': '어르신', '노부부': '어르신', '할머니': '어르신', '할배': '어르신',

    # 대처 및 해결책
    '관리실': '관리사무소', '관리소': '관리사무소', '관리사무실': '관리사무소', '방재실': '관리사무소',
    '경비실': '관리사무소', '경비': '관리사무소',
    '층간소음매트': '매트', '시공매트': '매트', '롤매트': '매트', '폴더매트': '매트', '퍼즐매트': '매트',
    '슬리퍼': '실내화', '덧신': '실내화', '층간소음슬리퍼': '실내화',

    # 상태/감정
    '노이로제': '스트레스', '빡침': '스트레스', '짜증': '스트레스', '분노': '스트레스', '환장': '스트레스',
    '귀트임': '소음민감', '귀 트임': '소음민감',

    # 주거 형태
    '아팟': '아파트'
층
}

# 동의어 치환 함수
def replace_synonyms(text):
    if not isinstance(text, str): return ""
    for key, value in synonym_dict.items():
        text = text.replace(key, value)
    return text

# [실행] 동의어 처리 진행
df['통합본문'] = df['통합본문'].apply(replace_synonyms)

# 4. 연속 공백 정리 (치환 후 발생한 공백 제거)
df['통합본문'] = df['통합본문'].str.replace(r'\s+', ' ', regex=True).str.strip()

print("✅ 동의어 및 유의어 통합 처리 완료!")

df.head(20)

# 대소문자 통일

In [ ]:
df["통합본문"] = df["통합본문"].apply(lambda x : x.lower())
df

#노이즈 제거

In [ ]:
# 2. 추가 노이즈 단어 리스트 (알바, 견적, 정보요청 등)
EXTRA_NOISE_WORDS = ['알바', '견적', '정보좀주세요', '공유부탁', '쪽지주세요', '비댓', '비댓글', '관상', '기아', '야구', '채용', '장투', '한화오션', '팬션','마스크',
                     '단가하락', '강원도', '빌라', '풀옵션', '동유럽', '패키지','미장','환율','오픈합니다','아파트','스푼라디오','법인택시', '김빈우', '발사이즈','변비','화장실']
noise_pattern = '|'.join(EXTRA_NOISE_WORDS)

# [실행] 노이즈 행 제거
before_len = len(df)
df = df[~df['통합본문'].str.contains(noise_pattern, na=False, case=False)]

In [ ]:
df

# 특수문자 및 이모지 제거

In [ ]:
import re

# 1. 한글(가-힣), 영문(a-zA-Z), 숫자(0-9), 공백(\s)을 제외한 모든 문자를 공백(' ')으로 대체
# 특수문자가 있던 자리를 띄어쓰기로 바꿔주어 단어들이 달라붙는 것을 방지합니다. (예: "층간소음/스트레스" -> "층간소음 스트레스")
df['통합본문'] = df['통합본문'].str.replace(r'[^가-힣a-zA-Z0-9\s]', ' ', regex=True)

# 2. ㅋㅋ, ㅎㅎ, ㅠㅠ 같은 단일 자음/모음도 노이즈가 될 수 있으므로 제거 (선택 사항)
df['통합본문'] = df['통합본문'].str.replace(r'[ㄱ-ㅎㅏ-ㅣ]', ' ', regex=True)

# 3. 특수문자가 공백으로 변환되면서 생긴 2개 이상의 연속된 공백을 하나로 줄이기
df['통합본문'] = df['통합본문'].str.replace(r'\s+', ' ', regex=True)

# 4. 문자열 양쪽 끝에 남아있는 공백 제거
df['통합본문'] = df['통합본문'].str.strip()
df

In [ ]:
# '통합본문_정제' 열 삭제
df = df.drop(columns=['통합본문_정제'])

df

#토크나이징

In [ ]:
okt = Okt()
df["token"] = df["통합본문"].apply(lambda x : okt.morphs(x, stem = True, norm = True))
df

In [ ]:
from tqdm import tqdm

okt = Okt()
tqdm.pandas()

def extract_pos(text, pos):
  left=[]
  for i in okt.pos(text,stem=True, norm =True):  # i 가 하나의 튜플 형성
    if i[1] in pos:   # 만약 i의 첫번째 원소가 명사 동사 형용사라면,
      left.append(i[0])
  return left

df["token"] = df["통합본문"].progress_apply(lambda x : extract_pos(x, ["Noun", "Verb", "Adjective"]))
df

# 불용어제거

In [ ]:
from kiwipiepy.utils import Stopwords

stopwords = Stopwords()
sw = set([i[0] for i in stopwords.stopwords]) # 중복 제거 위해 set 사용

cleaned_token = []
for i in df["token"]:
  imsi = []
  for w in i:
    if w not in sw:
      imsi.append(w)
  cleaned_token.append(imsi)
df["token"] = cleaned_token
df

In [ ]:
# 결과를 CSV 파일로 저장 (한글 깨짐 방지 및 인덱스 제외)
file_name = "레몬테라스_데이터전처리완료.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")
